# Encrypted Network Traffic Classifier with Deep Learning

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy import stats

np.random.seed(42)
types = ['HTTPS', 'VPN', 'TOR', 'SSH', 'Malware', 'Botnet']
data = []

for _ in range(3000):
    t = np.random.choice(types)
    if t == 'HTTPS': sizes, intervals = np.random.lognormal(7, 0.5, 50), np.random.exponential(0.1, 50)
    elif t == 'VPN': sizes, intervals = np.random.lognormal(6.5, 0.3, 50), np.random.exponential(0.05, 50)
    elif t == 'TOR': sizes, intervals = np.random.lognormal(6, 0.8, 50), np.random.exponential(0.2, 50)
    elif t == 'SSH': sizes, intervals = np.random.lognormal(5.5, 0.2, 50), np.random.exponential(0.3, 50)
    elif t == 'Malware': sizes, intervals = np.random.lognormal(7.5, 1.2, 50), np.random.exponential(0.01, 50)
    else: sizes, intervals = np.random.lognormal(6.8, 0.9, 50), np.random.exponential(0.005, 50)

    features = [np.mean(sizes), np.std(sizes), stats.skew(sizes), np.mean(intervals),
               np.std(intervals), np.sum(sizes), np.std(intervals)/np.mean(intervals),
               -np.sum(sizes/np.sum(sizes)*np.log2(sizes/np.sum(sizes)+1e-10))]
    data.append(features + [t])

X = np.array([row[:-1] for row in data], dtype=float)
y = np.array([row[-1] for row in data])

scaler, encoder = StandardScaler(), LabelEncoder()
X = scaler.fit_transform(X)
y = encoder.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential([Dense(128, activation='relu', input_shape=(8,)), Dropout(0.3),
                   Dense(64, activation='relu'), Dense(len(np.unique(y)), activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

accuracy = model.evaluate(X_test, y_test, verbose=0)[1]
predictions = model.predict(X_test)
print(f"Accuracy: {accuracy:.4f}")
print("Classes:", encoder.classes_)
print("Sample predictions:", encoder.inverse_transform(np.argmax(predictions[:5], axis=1)))